# Training dataframes, leaflet by leaflet

Builds the training tables for **LR, XGB and NN**, kept separate per leaflet so that one leaflet can be used
for training and the other for testing.

- Ld class (label 0): `dopc_pure`. Lo class (label 1): `dopc_60` (DOPC only).
- Features per chain: C1 … C16 order parameters, `xy_radius`, `z_height` (+ `label` in the last column).
- Rows are **lipid-major**: row = lipid × frames + frame, so each lipid's time series is one block of rows.
- Smoothing (`n`-pt) is a running average of **one lipid over n consecutive frames** (`convolve_features_valid`).

Output files in `heavy_data_NO_GIT/dataframes/sym/`:

| file | contents |
|---|---|
| `pure_dopc_{chain}_{leaflet}_op_ft_raw.csv`, `dopc_60_{chain}_{leaflet}_op_ft_raw.csv` | raw features per system (cache, built once) |
| `training_{chain}_{leaflet}_ft_raw.csv` | raw, pure + 60 stacked |
| `training_{chain}_{leaflet}_ft_{n}pt.csv` | n-pt smoothed, pure + 60 stacked |

`chain` = sn1 / sn2, `leaflet` = lower / upper. These are the names `NN_training`, `LR_training` and `XGB_training` read.

## 1. Settings

In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

import os
import time
import numpy as np
import pandas as pd

from code_libraries.bilayer_operations import Bilayer
from code_libraries.acyl_chain_preprocessing import op_height_df, convolve_features_valid, DOPC_op_H_av

start_frame, stop_frame = 0, 4000        # trajectories have 4001 frames (300-500 ns, 50 ps apart)
n_frames = stop_frame - start_frame

CHAINS   = ["sn1", "sn2"]
LEAFLETS = ["lower", "upper"]

# windows to build ("raw" = no smoothing). Must match the windows the training notebooks use.
PT_SIZES = ["raw", 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 51, 101, 151, 201]

REBUILD_RAW = False   # True: recompute the raw feature cache even if the files exist
OVERWRITE   = False   # True: rebuild the smoothed training files even if they exist

dataframe_dir = os.path.join("../", "heavy_data_NO_GIT/dataframes/sym/")
os.makedirs(dataframe_dir, exist_ok=True)

# system name -> (file prefix, is_pure)
SYSTEMS = {"dopc_pure": ("pure_dopc", True),
           "dopc_60":   ("dopc_60",   False)}

## 2. Load the leaflets

Positions come from `heavy_data_NO_GIT/position_npys/sym/` (made by `npy_array_generation`).
For `dopc_60` the leaflet is a tuple (DOPC, CHOL); only the DOPC part is used.
Each leaflet is used with **its own** data (this is where the old sn2 bug was: `dopc_60` sn2 "upper" was built from the lower leaflet).

In [ ]:
leaflets = {}    # (system, leaflet) -> array (frames, lipids, 138 atoms, 3)

for sys_name in SYSTEMS:
    bil = Bilayer(sys_name, "sym", start_frame, stop_frame)
    bil.cleave_leaflets()
    for leaflet, data in (("lower", bil.lower_leaflet), ("upper", bil.upper_leaflet)):
        dopc = data[0] if isinstance(data, tuple) else data
        assert dopc.shape[0] == n_frames, f"{sys_name} {leaflet}: {dopc.shape[0]} frames, expected {n_frames}"
        leaflets[(sys_name, leaflet)] = dopc
        print(f"{sys_name:10s} {leaflet}: {dopc.shape[1]} DOPC, {dopc.shape[0]} frames")

# the two leaflets of one system must be different molecules
for sys_name in SYSTEMS:
    assert not np.array_equal(leaflets[(sys_name, "lower")][0], leaflets[(sys_name, "upper")][0])

## 3. Raw features per system, chain and leaflet (slow: run once)

Each file takes a few minutes. Existing files are reused unless `REBUILD_RAW = True`.
The check after each one confirms the rows are lipid-major: row 1 must be lipid 0 at frame 1.

In [ ]:
def raw_path(sys_name, chain, leaflet):
    prefix = SYSTEMS[sys_name][0]
    return os.path.join(dataframe_dir, f"{prefix}_{chain}_{leaflet}_op_ft_raw.csv")

raw = {}   # (system, chain, leaflet) -> dataframe

for sys_name, (prefix, is_pure) in SYSTEMS.items():
    for chain in CHAINS:
        for leaflet in LEAFLETS:
            p = raw_path(sys_name, chain, leaflet)
            if os.path.exists(p) and not REBUILD_RAW:
                df = pd.read_csv(p)
                print(f"loaded  {os.path.basename(p)}  {df.shape}")
            else:
                t = time.time()
                df = op_height_df(leaflets[(sys_name, leaflet)], chain, n_frames, is_pure=is_pure, heights=True)
                df.to_csv(p, index=False)
                print(f"built   {os.path.basename(p)}  {df.shape}  ({time.time() - t:.0f} s)")

            # checks: size, label, lipid-major row order
            n_lip = leaflets[(sys_name, leaflet)].shape[1]
            assert df.shape[0] == n_lip * n_frames
            assert (df["label"] == (0 if is_pure else 1)).all()
            op_lipid0_frame1 = DOPC_op_H_av(leaflets[(sys_name, leaflet)][1, 0], chain)
            assert np.allclose(df.iloc[1, :16].to_numpy(dtype=float), op_lipid0_frame1), \
                f"{os.path.basename(p)} is not lipid-major (made with the old op_height_df?) - set REBUILD_RAW = True"
            raw[(sys_name, chain, leaflet)] = df

## 4. Training files for every window

For each window, each system is smoothed lipid by lipid, then pure (label 0) and 60 (label 1) are stacked:
pure rows first, then 60. Existing files are skipped unless `OVERWRITE = True`.

In [ ]:
def training_path(chain, leaflet, pt):
    suffix = "raw" if pt == "raw" else f"{pt}pt"
    return os.path.join(dataframe_dir, f"training_{chain}_{leaflet}_ft_{suffix}.csv")

t_all = time.time()
for pt in PT_SIZES:
    for chain in CHAINS:
        for leaflet in LEAFLETS:
            out = training_path(chain, leaflet, pt)
            if os.path.exists(out) and not OVERWRITE:
                print(f"exists  {os.path.basename(out)}")
                continue
            parts = []
            for sys_name, (prefix, is_pure) in SYSTEMS.items():
                df = raw[(sys_name, chain, leaflet)]
                if pt != "raw":
                    n_red = n_frames - (pt - 1)          # frames left per lipid after an n-pt running average
                    df = convolve_features_valid(df, n_red, n_frames, label_type="pure" if is_pure else "not pure")
                parts.append(df)
            combined = pd.concat(parts, axis=0, ignore_index=True)
            combined.to_csv(out)                         # with index column, as NN_training expects ("Unnamed: 0")
            print(f"written {os.path.basename(out)}  {combined.shape}  "
                  f"(Ld rows {int((combined['label'] == 0).sum())}, Lo rows {int((combined['label'] == 1).sum())})")
print(f"done in {(time.time() - t_all) / 60:.1f} min")

## 5. Check: the smoothing is over time, within one lipid

Lipid 0 of `dopc_pure` (sn1, lower): row t of the n-pt file must equal the mean of that lipid's frames t … t+n-1.

In [ ]:
pt_check = next(p for p in PT_SIZES if p != "raw")
sm = pd.read_csv(training_path("sn1", "lower", pt_check), index_col=0)
rw = raw[("dopc_pure", "sn1", "lower")]
for t in (0, 1, n_frames - pt_check):          # first, second and last smoothed frame of lipid 0
    expected = rw.iloc[t:t + pt_check, :18].mean().to_numpy()
    assert np.allclose(sm.iloc[t, :18].to_numpy(), expected), t
print(f"OK: {pt_check}-pt rows are time averages of one lipid")